# Jour 2 · Convolution et premier CNN

> **Version corrigée** — les cellules de solution sont signalées.

## Objectifs

- comprendre filtre, convolution, carte de caractéristiques et pooling
- entraîner un petit CNN sur les mêmes images
- expliquer pourquoi les CNN conviennent mieux aux structures spatiales


## Le problème du réseau Dense

En aplatissant l'image, nous traitons les pixels comme une longue liste. Un **CNN**, ou réseau convolutif, conserve la disposition spatiale et cherche de petits motifs locaux : bords, orientations, textures puis formes plus complexes.


![Fenêtre de convolution glissant sur une image](../../ressources/illustrations/jour_02/03_convolution.png)

*Un même filtre est appliqué partout ; ses poids sont appris pendant l’entraînement.*


## Les termes essentiels

- Un **filtre** ou *kernel* est une petite grille de poids.
- La **convolution** fait glisser ce filtre sur l'image et calcule une réponse à chaque position.
- Une **feature map** est la carte des réponses produites par un filtre.
- Le **pooling** réduit la taille spatiale en conservant une information résumée.

Nous n'avons pas besoin de calculer chaque multiplication à la main. L'idée importante est le partage des poids : le même détecteur de motif cherche à plusieurs endroits.

### Lire l'architecture Keras

`Conv2D(16, kernel_size=3)` apprend 16 filtres de taille 3 × 3. `padding="same"` ajoute une bordure technique afin de conserver ici la largeur et la hauteur après la convolution. La couche suivante apprend 32 filtres : elle peut combiner les motifs détectés précédemment.

`MaxPooling2D(pool_size=2)` divise approximativement la largeur et la hauteur par deux en conservant la réponse la plus forte de chaque petite zone. Le nombre placé à la fin de la forme d'un tenseur correspond aux **canaux** ou cartes de caractéristiques.



![Hiérarchie des motifs appris par un CNN](../../ressources/illustrations/jour_02/03_hierarchie_motifs.png)

Les premières couches reçoivent directement les pixels et apprennent souvent des motifs locaux simples. Les couches suivantes combinent leurs feature maps pour construire des représentations plus riches.

Cette description aide à comprendre, mais le réseau ne reçoit pas les mots « bord » ou « courbe ». Il ajuste des nombres parce que certaines réponses de filtres contribuent à réduire la loss.

![Évolution des dimensions dans le CNN](../../ressources/illustrations/jour_02/03_pooling_dimensions.png)

Avec `padding="same"` et un pas de convolution égal à 1, la hauteur et la largeur sont conservées. Les 16 filtres produisent 16 feature maps. Le pooling 2 × 2 réduit ensuite 8 × 8 à 4 × 4 tout en gardant les 16 canaux. La seconde convolution produit 32 cartes.

Suivre les dimensions couche par couche permet de détecter beaucoup d’erreurs d’architecture avant même l’entraînement.


![Pipeline image convolution pooling dense softmax](../../ressources/illustrations/jour_02/03_architecture_cnn.png)

*Les premières couches recherchent des motifs ; les dernières combinent ces motifs pour choisir une classe.*


## Régulariser, enrichir et réutiliser

![Moyens de favoriser la généralisation d'un réseau](../../ressources/illustrations/jour_02/03_regularisation_generalisation.png)

La **régularisation** regroupe les techniques qui cherchent à améliorer le comportement sur de nouvelles données, pas seulement sur les exemples d'entraînement.

- L'**augmentation de données** fabrique à la volée des variantes plausibles : petite rotation, translation, recadrage ou changement de luminosité. La transformation doit préserver la bonne réponse.
- Le **Dropout** empêche certains neurones de devenir indispensables.
- L'**arrêt anticipé** conserve les poids obtenus avant que la validation ne se dégrade.
- Le **transfert d'apprentissage** part d'un réseau déjà entraîné. On garde ses représentations utiles et on adapte une nouvelle tête de sortie.

Ici, de petites rotations et translations suffisent pour illustrer l'augmentation. Sur de vraies images, les transformations doivent toujours rester réalistes pour le problème traité.


### Pourquoi les transformations doivent rester plausibles

L’augmentation est active uniquement pendant l’entraînement. Chaque passage peut présenter une version légèrement différente de la même image, mais le label reste inchangé.

Une transformation utile dépend du problème. Une petite translation d’un chiffre reste plausible ; une rotation complète peut transformer son sens. Pour certains objets, un retournement horizontal est acceptable, alors que pour du texte il peut créer une donnée impossible.

L’objectif n’est donc pas de déformer au maximum, mais de représenter les variations normales que le modèle rencontrera.


In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass
plt.style.use("seaborn-v0_8-whitegrid")
print("TensorFlow", tf.__version__)

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

digits = load_digits()
images = digits.images.astype("float32")
labels = digits.target.astype("int64")
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, stratify=labels, random_state=42
)
X_train = X_train / 16.0
X_test = X_test / 16.0
print("Train :", X_train.shape, "Test :", X_test.shape)

X_train_cnn = X_train[..., np.newaxis]
X_test_cnn = X_test[..., np.newaxis]
print("Forme CNN :", X_train_cnn.shape)


## Pourquoi ajouter un axe de canal ?

Les images `digits` sont stockées sous la forme `(nombre, 8, 8)`. `Conv2D` attend aussi un axe de canaux, même pour une image en niveaux de gris.

L’écriture `X_train[..., np.newaxis]` conserve toutes les dimensions existantes et ajoute un axe de taille 1 à la fin. La forme devient donc `(nombre, 8, 8, 1)`.

Ce `1` ne signifie pas qu’il existe une seule image. Il signifie que chaque image possède un seul canal d’intensité. Une image RGB utiliserait généralement trois canaux.


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.04),
    tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
], name="augmentation")

cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8, 1)),
    augmentation,
    tf.keras.layers.Conv2D(16, kernel_size=3, padding="same", activation="relu", name="conv_1"),
    tf.keras.layers.MaxPooling2D(pool_size=2),
    tf.keras.layers.Conv2D(32, kernel_size=3, padding="same", activation="relu", name="conv_2"),
    tf.keras.layers.Flatten(name="features"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

arret_anticipe = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = cnn.fit(
    X_train_cnn, y_train,
    validation_split=0.20,
    epochs=20,
    batch_size=32,
    callbacks=[arret_anticipe],
    verbose=0,
)
print("Époques réellement effectuées :", len(history.history["loss"]))
cnn.summary()


In [ ]:
history_df = pd.DataFrame(history.history)
history_df[["accuracy", "val_accuracy"]].plot(figsize=(8, 3), title="Accuracy du CNN")
plt.xlabel("époque")
plt.show()

loss, accuracy = cnn.evaluate(X_test_cnn, y_test, verbose=0)
print("Accuracy test du CNN :", round(accuracy, 3))


## Mini-démonstration de transfert d'apprentissage

Notre CNN sait maintenant extraire des caractéristiques utiles pour reconnaître les chiffres. Nous allons **geler** cet extracteur — ses poids ne changeront plus — puis remplacer sa tête de sortie.

La nouvelle tâche est différente : prédire si le chiffre est **pair ou impair**. Dans un projet réel, on réutilise souvent un modèle pré-entraîné sur un jeu beaucoup plus vaste. Cette petite expérience reproduit le même mécanisme à notre échelle.


![Principe du transfert d’apprentissage](../../ressources/illustrations/jour_02/03_transfert_apprentissage.png)

L’**extracteur** correspond ici aux couches situées avant la tête de classification. Il transforme une image en un vecteur de caractéristiques. En fixant `extracteur.trainable = False`, on interdit à l’optimiseur de modifier ses poids.

La nouvelle couche Dense constitue la **tête** adaptée à la nouvelle cible pair/impair. Elle est la seule partie entraînée dans cette démonstration.

Dans un véritable transfert, le modèle source a souvent appris sur beaucoup plus de données. On commence généralement par entraîner la nouvelle tête avec l’extracteur gelé. Le **fine-tuning** consiste ensuite à dégeler quelques couches et à les ajuster prudemment avec un petit learning rate.


In [ ]:
extracteur = tf.keras.Model(
    inputs=cnn.inputs,
    outputs=cnn.get_layer("features").output,
    name="extracteur_figures",
)
extracteur.trainable = False

y_train_parite = (y_train % 2 == 0).astype("float32")
y_test_parite = (y_test % 2 == 0).astype("float32")

modele_parite = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8, 1)),
    extracteur,
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
modele_parite.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
modele_parite.fit(
    X_train_cnn,
    y_train_parite,
    validation_split=0.20,
    epochs=6,
    batch_size=32,
    verbose=0,
)

loss_parite, accuracy_parite = modele_parite.evaluate(
    X_test_cnn, y_test_parite, verbose=0
)
print("Couches de l'extracteur réentraînables :", len(extracteur.trainable_weights))
print("Accuracy pair / impair :", round(accuracy_parite, 3))


La valeur `0` pour les poids réentraînables confirme que l'extracteur est gelé. Seule la nouvelle couche de sortie apprend. On peut aussi dégeler une partie du réseau et poursuivre avec un très petit pas d'apprentissage : cette étape s'appelle le **fine-tuning**.


## Ce que ce petit CNN apprend — et ce qu’il ne garantit pas

Le réseau reçoit une image contenant un seul chiffre déjà cadré et produit une classe parmi dix. Il apprend des motifs utiles sur ce jeu de données précis. Une bonne accuracy sur le test signifie qu’il généralise raisonnablement à des images construites de la même manière.

Elle ne garantit pas son comportement sur une photo couleur, un chiffre tourné, une autre résolution ou une écriture très différente. Ce changement entre les données apprises et les données rencontrées est appelé **décalage de distribution**.

Ajouter des couches n’améliore pas automatiquement un modèle. Une architecture plus grande augmente sa capacité, son temps de calcul et son risque de mémoriser le train. On choisit donc une architecture en comparant ses résultats, ses erreurs, son coût et les contraintes du problème.

### Pourquoi les CNN sont efficaces sur les images

Le même filtre est réutilisé à toutes les positions : c’est le **partage des poids**. Le réseau apprend ainsi un détecteur local une seule fois et peut le rechercher partout. Cette propriété demande beaucoup moins de paramètres qu’une connexion indépendante pour chaque motif et chaque position.


## À vous de jouer — lire les dimensions

Affichez la forme de la sortie de la première couche convolutionnelle pour un batch de quatre images.


In [ ]:
modele_feature_maps = tf.keras.Model(
    inputs=cnn.inputs,
    outputs=cnn.get_layer("conv_1").output,
)
feature_maps = modele_feature_maps.predict(X_test_cnn[:4], verbose=0)
print("Entrée :", X_test_cnn[:4].shape)
print("Sortie de la première convolution :", feature_maps.shape)
print("Interprétation : 4 images, 8 × 8 positions, 16 cartes de caractéristiques.")
